<a href="https://colab.research.google.com/github/amol004/IndustryGPT-Specialised-LLM-Bot-Using-Pre-Trained-Models/blob/main/IndustryGPT_Specialized_LLM_Bot_Using_Pre_Trained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Installing Required Libraries
# fine-tuned LLM without requiring large GPU infrastructure.
!pip install -q transformers datasets peft bitsandbytes accelerate trl

In [2]:
import torch  # Loads the core deep learning library for tensor calculations
from datasets import load_dataset  # Imports the tool used to fetch data from Hugging Face
from transformers import AutoTokenizer  # Imports the tool to turn text into numbers
from transformers import AutoModelForCausalLM  # Imports the core for our model architecture
from transformers import TrainingArguments  # Imports the setup for training parameters
from transformers import Trainer  # Imports the engine that runs the training loop
from transformers import BitsAndBytesConfig  # Imports tools for model memory compression
from peft import LoraConfig  # Imports settings for efficient adapter-based tuning
from peft import get_peft_model  # Imports the function to apply adapters to the model

## **Dataset loading**

In [3]:
# Import necessary tools
from huggingface_hub import login  # Tool to sign in to Hugging Face
from google.colab import userdata  # Tool to get your private API keys
from datasets import load_dataset   # Tool to download AI datasets

# Sign in using the secret token
hf_token = userdata.get('HF_TOKEN')  # Pull the key named 'HF_TOKEN' from Colab secrets
login(token=hf_token)                # Send that key to Hugging Face to log in

# Download the retail dataset from huggingface
data_path = "bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset"
dataset = load_dataset(data_path)     # Download the specific e-commerce dataset

# Show what we downloaded
print(dataset)                        # Display the data structure and row counts

README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

bitext-retail-ecommerce-llm-chatbot-trai(…):   0%|          | 0.00/42.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44884 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'intent', 'category', 'tags', 'response'],
        num_rows: 44884
    })
})


## **Dataset first look**

In [4]:
# Display the first few rows
# Convert the first 5 rows of the 'train' split to a pandas DataFrame for a nice table view
import pandas as pd

top_rows = dataset['train'].select(range(5))
display(pd.DataFrame(top_rows))

,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...


### Why we chose this Dataset

*   **Industry Specific**: Focuses on retail and e-commerce, making it perfect for customer service bots.
*   **Large Scale**: With over 44,000 rows, it provides enough diversity for the model to generalize well.
*   **Structured for LLMs**: The `instruction` and `response` format is exactly what modern LLMs need for supervised fine-tuning (SFT).
*   **High Quality**: The responses are clear, professional, and follow a consistent tone.

## **PREPARE AND CLEAN THE DATA**

In [5]:
# Define a function to merge instructions and responses into one 'text' field
def create_prompt_format(row):
    # We combine the user input and the bot answer into a standardized template
    prompt_template = f"### Instruction:\n{row['instruction']}\n\n### Response:\n{row['response']}"
    return {"text": prompt_template}

# Apply the formatting function to every row in the training set
dataset = dataset["train"].map(create_prompt_format)

# Keep only the 'text' column and discard metadata we no longer need for training
columns_to_drop = [name for name in dataset.column_names if name != "text"]
dataset = dataset.remove_columns(columns_to_drop)

# Confirmation of the new structure
print("Data formatting complete. Ready for fine-tuning.")

Map:   0%|          | 0/44884 [00:00<?, ? examples/s]

Data formatting complete. Ready for fine-tuning.


In [6]:
# Let's print the first row of our newly formatted dataset to see the final structure
print("Sample of formatted text column:")  # Prints a header title for the output
print("-" * 30)  # Prints a visual separator line using 30 dashes
print(dataset[0]['text'])  # Accesses the first record [0] and displays the 'text' field content
print("-" * 30)  # Prints a visual closing separator line

Sample of formatted text column:
------------------------------
### Instruction:
I got to add an item to the cart

### Response:
I'll get right on it! I'm here to assist you in adding an item to your cart. Here's a step-by-step guide to help you out:

1. Browse through our {{Product Catalog}} to find the item you want to add to your cart.
2. Once you've found the item, click on the "{{Add to Cart}}" button or select the desired quantity from the available options.
3. You'll be redirected to your cart page, where you can review your selected items.
4. If you want to continue shopping, simply click on the "{{Continue Shopping}}" button, and you'll be taken back to the catalog.
5. If you're ready to proceed, click on the "{{Checkout}}" button to finalize your purchase.

If you encounter any difficulties during this process, please don't hesitate to let me know. I'm here to help you with any questions or concerns you may have. Happy shopping!
------------------------------


In [7]:
# --- CHECK GPU SPECIFICATIONS ---
# This command checks the details of the available GPU hardware
!nvidia-smi  # Execute the NVIDIA System Management Interface to display GPU specs

Thu Jun 11 18:37:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----